# 05 — Scoring en Vivo

Aplica el modelo entrenado sobre mercados **activos** para generar señales de compra.

**Prerequisitos:**
```bash
# Modelo entrenado
python -m src.model.train --data-dir data/processed --epochs 50

# O desde el notebook 04_model_training.ipynb
```

**Secciones:**
1. Cargar modelo y pipeline
2. Puntuar mercados activos (de disco o en tiempo real)
3. Top-K señales con clasificación STRONG BUY / BUY / HOLD
4. Filtros de calidad (liquidez, spread, volumen)
5. Backtesting sobre resueltos (validación de la señal)
6. Simulación de P&L

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

MODELS_DIR  = ROOT / 'data' / 'models'
PROCESSED   = ROOT / 'data' / 'processed'
RAW         = ROOT / 'data' / 'raw'

# Umbrales de señal (de config/config.yaml)
BUY_THRESHOLD        = 0.60
STRONG_BUY_THRESHOLD = 0.75
MIN_LIQUIDITY        = 1_000.0
MIN_VOLUME_24H       = 100.0
MAX_SPREAD           = 0.10

model_path = MODELS_DIR / 'best_market_model.pt'
pipeline_dir = PROCESSED / 'pipeline'

assert model_path.exists(), (
    f"Modelo no encontrado: {model_path}\n"
    "Entrenar primero con: python -m src.model.train  o con el notebook 04"
)
assert pipeline_dir.exists(), (
    f"Pipeline no encontrado: {pipeline_dir}\n"
    "Correr primero: python -m src.features.pipeline"
)
print("Modelo y pipeline disponibles.")

## 1. Cargar modelo y pipeline

In [ ]:
from src.features.pipeline import FeaturePipeline
from src.model.architecture import MarketValueNet

# Cargar pipeline (scaler + category encoder, ya fitted)
pipeline = FeaturePipeline.load(str(pipeline_dir), use_dummy_text=False)

# Cargar modelo con las dimensiones del pipeline
model = MarketValueNet(
    num_numerical_features=pipeline.num_numerical_features,
    num_categories=pipeline.num_categories,
    text_embed_dim=pipeline.text_embed_dim,
    hidden_dims=[256, 128, 64],
    dropout=0.3,
    task="classification",
)
model.load_state_dict(torch.load(model_path, weights_only=True))
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Modelo cargado desde {model_path.name}")
print(f"  num_numerical_features: {pipeline.num_numerical_features}")
print(f"  num_categories:         {pipeline.num_categories}")
print(f"  text_embed_dim:         {pipeline.text_embed_dim}")
print(f"  Dispositivo: {device}")

## 2. Puntuar mercados activos

Usamos los mercados activos descargados en `data/raw/active_markets.json`.
Para scoring en tiempo real, usar `python -m src.scoring.scorer --top 20`.

In [ ]:
with open(RAW / 'active_markets.json') as f:
    active_markets = json.load(f)
with open(RAW / 'price_histories.json') as f:
    price_histories = json.load(f)

print(f"Mercados activos cargados: {len(active_markets):,}")

results = []
skipped = 0

for market in active_markets:
    mid = str(market.get('id', ''))
    hist = price_histories.get(mid)
    try:
        ob = None  # order book no disponible en este notebook; scorer.py lo descarga en vivo
        features = pipeline.transform_single(market, price_history=hist, order_book=ob)

        num_t = torch.FloatTensor(features['numerical']).unsqueeze(0).to(device)
        cat_t = torch.LongTensor([features['category_id']]).to(device)
        txt_t = torch.FloatTensor(features['text_embedding']).unsqueeze(0).to(device)

        with torch.no_grad():
            score = model(num_t, cat_t, txt_t).item()

        op = market.get('outcomePrices', [0.5])
        price_yes = float(op[0]) if isinstance(op, list) and op else float(market.get('lastTradePrice') or 0.5)

        results.append({
            'id':          mid,
            'question':    market.get('question', ''),
            'price_yes':   price_yes,
            'volume_24h':  float(market.get('volume24hr') or 0),
            'liquidity':   float(market.get('liquidity') or 0),
            'spread':      float(market.get('spread') or 0),
            'model_score': score,
            'end_date':    market.get('endDate', ''),
            'slug':        market.get('slug', ''),
        })
    except Exception:
        skipped += 1

df_scores = pd.DataFrame(results).sort_values('model_score', ascending=False).reset_index(drop=True)
print(f"Puntuados: {len(df_scores):,} | Omitidos: {skipped}")
print(f"\nDistribución de scores:")
print(df_scores['model_score'].describe().round(4))

## 3. Top-K señales con clasificación

El `model_score` es un score de clasificación, **no una probabilidad calibrada**.
Usarlo como **ranking relativo** entre mercados, no como retorno esperado.

| Score | Señal |
|---|---|
| ≥ 0.75 | STRONG BUY |
| ≥ 0.60 | BUY |
| < 0.60 | HOLD |

In [ ]:
def classify_signal(score):
    if score >= STRONG_BUY_THRESHOLD:
        return 'STRONG BUY'
    elif score >= BUY_THRESHOLD:
        return 'BUY'
    else:
        return 'HOLD'

df_scores['signal'] = df_scores['model_score'].apply(classify_signal)
df_scores['score_minus_price'] = df_scores['model_score'] - df_scores['price_yes']

signal_counts = df_scores['signal'].value_counts()
print("Distribución de señales:")
for sig, cnt in signal_counts.items():
    print(f"  {sig:12s}: {cnt:,} ({100*cnt/len(df_scores):.1f}%)")

print(f"\n{'─'*100}")
print(f"{'#':>3} {'SEÑAL':12} {'SCORE':6} {'PRECIO':7} {'VOL24H':>10} {'QUESTION'}")
print(f"{'─'*100}")
for i, (_, row) in enumerate(df_scores[df_scores['signal'] != 'HOLD'].head(20).iterrows(), 1):
    q = row['question'][:60]
    print(f"{i:>3} {row['signal']:12} {row['model_score']:.3f}  ${row['price_yes']:.2f}  ${row['volume_24h']:>9,.0f}  {q}")

## 4. Filtros de calidad

Aplicar criterios mínimos de liquidez para reducir slippage y ruido en mercados poco líquidos.

In [ ]:
df_filtered = df_scores[
    (df_scores['liquidity']  >= MIN_LIQUIDITY) &
    (df_scores['volume_24h'] >= MIN_VOLUME_24H) &
    (df_scores['spread']     <= MAX_SPREAD) &
    (df_scores['signal']     != 'HOLD')
].copy()

print(f"Señales antes de filtrar: {len(df_scores[df_scores['signal'] != 'HOLD']):,}")
print(f"Señales después de filtros de calidad: {len(df_filtered):,}")
print(f"  Filtros: liquidez≥${MIN_LIQUIDITY:,.0f}, vol24h≥${MIN_VOLUME_24H:,.0f}, spread≤{MAX_SPREAD:.0%}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.hist(df_scores['model_score'], bins=50, color=PALETTE[0], alpha=0.7, label='Todos')
ax.axvline(BUY_THRESHOLD, color=PALETTE[2], linestyle='--', lw=1.5, label=f'Buy ({BUY_THRESHOLD})')
ax.axvline(STRONG_BUY_THRESHOLD, color=PALETTE[3], linestyle='--', lw=1.5, label=f'Strong ({STRONG_BUY_THRESHOLD})')
ax.set_title('Distribución de scores')
ax.set_xlabel('Model score')
ax.legend()

ax = axes[1]
signal_order = ['HOLD', 'BUY', 'STRONG BUY']
colors_map = {'HOLD': PALETTE[0], 'BUY': PALETTE[2], 'STRONG BUY': PALETTE[3]}
for sig in signal_order:
    sub = df_scores[df_scores['signal'] == sig]['model_score']
    if len(sub) > 0:
        ax.hist(sub, bins=30, alpha=0.6, color=colors_map[sig], label=f'{sig} ({len(sub):,})', density=True)
ax.set_title('Scores por señal')
ax.set_xlabel('Model score')
ax.legend()

ax = axes[2]
ax.scatter(df_scores['price_yes'], df_scores['model_score'],
           c=[{'HOLD': PALETTE[0], 'BUY': PALETTE[2], 'STRONG BUY': PALETTE[3]}[s] for s in df_scores['signal']],
           alpha=0.4, s=20)
ax.axhline(BUY_THRESHOLD, color='gray', linestyle='--', lw=1)
ax.axhline(STRONG_BUY_THRESHOLD, color='black', linestyle='--', lw=1)
ax.set_xlabel('Precio Yes actual')
ax.set_ylabel('Model score')
ax.set_title('Score vs Precio')

plt.suptitle('Análisis de señales — mercados activos', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'scoring_signals.png', bbox_inches='tight')
plt.show()

## 5. Backtesting sobre mercados resueltos

Simulamos cómo habría rendido la señal del modelo en mercados que ya resolvieron.
Usamos el precio snapshot (7 días antes del endDate) como precio de entrada.

In [ ]:
from src.data.preprocessing import get_snapshot_price, _infer_resolution_from_market

with open(RAW / 'resolved_markets.json') as f:
    resolved_markets = json.load(f)

# Tomar una muestra para el backtest (evitar tardar demasiado con el text encoder)
BACKTEST_SAMPLE = 2000
sample = resolved_markets[:BACKTEST_SAMPLE]

backtest_rows = []
skipped_bt = 0

for market in sample:
    mid = str(market.get('id', ''))
    hist = price_histories.get(mid)
    snap = get_snapshot_price(market, price_histories, snapshot_offset_days=7)
    if snap is None:
        skipped_bt += 1
        continue

    # Inyectar snapshot como precio de entrada (anti-leakage)
    m_copy = market.copy()
    m_copy['outcomePrices'] = [snap, 1.0 - snap]

    try:
        features = pipeline.transform_single(m_copy, price_history=hist)
        num_t = torch.FloatTensor(features['numerical']).unsqueeze(0).to(device)
        cat_t = torch.LongTensor([features['category_id']]).to(device)
        txt_t = torch.FloatTensor(features['text_embedding']).unsqueeze(0).to(device)

        with torch.no_grad():
            score = model(num_t, cat_t, txt_t).item()

        resolution = _infer_resolution_from_market(market)
        if resolution not in ('yes', 'no'):
            continue

        backtest_rows.append({
            'id':         mid,
            'question':   market.get('question', '')[:60],
            'snap_price': snap,
            'score':      score,
            'signal':     classify_signal(score),
            'resolved':   1 if resolution == 'yes' else 0,
        })
    except Exception:
        skipped_bt += 1

df_bt = pd.DataFrame(backtest_rows)
print(f"Muestra backtest: {len(df_bt):,} mercados ({skipped_bt} omitidos)")
print(f"\n{'─'*50}")
print("Win rate por señal:")
for sig in ['STRONG BUY', 'BUY', 'HOLD']:
    sub = df_bt[df_bt['signal'] == sig]
    if len(sub) == 0:
        continue
    wr = sub['resolved'].mean()
    print(f"  {sig:12s}: {len(sub):>4,} trades | {100*wr:.1f}% Yes")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Win rate por señal
ax = axes[0]
sig_order = ['STRONG BUY', 'BUY', 'HOLD']
sig_colors = [PALETTE[3], PALETTE[2], PALETTE[0]]
wr_vals = []
counts = []
for sig in sig_order:
    sub = df_bt[df_bt['signal'] == sig]
    wr_vals.append(sub['resolved'].mean() if len(sub) > 0 else 0)
    counts.append(len(sub))
bars = ax.bar(sig_order, wr_vals, color=sig_colors, alpha=0.85, width=0.5)
for bar, wr, cnt in zip(bars, wr_vals, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{100*wr:.1f}%\n(n={cnt:,})', ha='center', fontsize=9)
ax.axhline(df_bt['resolved'].mean(), color='black', linestyle='--', lw=1.5, label=f'Base rate ({100*df_bt["resolved"].mean():.1f}%)')
ax.set_ylim(0, 1)
ax.set_ylabel('Win rate (% Yes)')
ax.set_title('Win rate por señal')
ax.legend()

# Score vs win rate (binned)
ax = axes[1]
df_bt['score_bin'] = pd.cut(df_bt['score'], bins=10)
wr_by_bin = df_bt.groupby('score_bin', observed=True)['resolved'].agg(['mean', 'count'])
bin_centers = [iv.mid for iv in wr_by_bin.index]
ax.bar(range(len(wr_by_bin)), wr_by_bin['mean'], color=PALETTE[0], alpha=0.7)
ax.axhline(df_bt['resolved'].mean(), color='red', linestyle='--', lw=1.5, label='Base rate')
ax.set_xticks(range(len(wr_by_bin)))
ax.set_xticklabels([f'{c:.2f}' for c in bin_centers], rotation=45, fontsize=8)
ax.set_xlabel('Score (bin)')
ax.set_ylabel('Win rate (% Yes)')
ax.set_title('Win rate vs score (10 bins)')
ax.legend()

plt.suptitle('Backtest — win rate de la señal en mercados resueltos', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'scoring_backtest_winrate.png', bbox_inches='tight')
plt.show()

## 6. Simulación de P&L

Simulamos P&L con position sizing fijo (5% del capital por trade).
Compramos Yes al precio snapshot cuando la señal es BUY o STRONG BUY.

In [ ]:
INITIAL_CAPITAL = 1_000.0
POSITION_SIZE   = 0.05  # 5% del capital por trade

# Filtrar solo señales de compra (sin HOLD) con precio válido
df_trades = df_bt[
    (df_bt['signal'].isin(['BUY', 'STRONG BUY'])) &
    (df_bt['snap_price'] > 0.01) &
    (df_bt['snap_price'] < 0.99)
].copy().reset_index(drop=True)

capital = INITIAL_CAPITAL
equity_curve = [capital]
trade_log = []

for _, row in df_trades.iterrows():
    bet = capital * POSITION_SIZE
    price = row['snap_price']
    shares = bet / price
    payout = shares * float(row['resolved'])  # 1 si Yes, 0 si No
    pnl = payout - bet
    capital += pnl
    equity_curve.append(capital)
    trade_log.append({
        'signal':    row['signal'],
        'price':     price,
        'resolved':  int(row['resolved']),
        'pnl':       pnl,
        'capital':   capital,
    })

df_log = pd.DataFrame(trade_log)
final_capital = capital
total_return = (final_capital / INITIAL_CAPITAL - 1) * 100
win_rate_bt = df_log['resolved'].mean()
avg_pnl = df_log['pnl'].mean()

print(f"Simulación P&L — {len(df_log)} trades")
print(f"  Capital inicial:  ${INITIAL_CAPITAL:,.2f}")
print(f"  Capital final:    ${final_capital:,.2f}")
print(f"  Retorno total:    {total_return:+.2f}%")
print(f"  Win rate:         {100*win_rate_bt:.1f}%")
print(f"  PnL promedio:     ${avg_pnl:+.2f} por trade")
print(f"  Trades ganadores: {df_log['resolved'].sum():.0f}")
print(f"  Trades perdedores:{(1-df_log['resolved']).sum():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Equity curve
ax = axes[0]
ax.plot(equity_curve, color=PALETTE[0], lw=2)
ax.axhline(INITIAL_CAPITAL, color='gray', linestyle='--', lw=1, label='Capital inicial')
ax.fill_between(range(len(equity_curve)), INITIAL_CAPITAL, equity_curve,
                where=[e >= INITIAL_CAPITAL for e in equity_curve],
                alpha=0.2, color=PALETTE[2], label='Ganancia')
ax.fill_between(range(len(equity_curve)), INITIAL_CAPITAL, equity_curve,
                where=[e < INITIAL_CAPITAL for e in equity_curve],
                alpha=0.2, color=PALETTE[3], label='Pérdida')
ax.set_title(f'Equity curve ({len(df_log)} trades)')
ax.set_xlabel('Trade #')
ax.set_ylabel('Capital ($)')
ax.legend()

# PnL por trade
ax = axes[1]
colors_pnl = [PALETTE[2] if p > 0 else PALETTE[3] for p in df_log['pnl']]
ax.bar(range(len(df_log)), df_log['pnl'], color=colors_pnl, alpha=0.7)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('PnL por trade')
ax.set_xlabel('Trade #')
ax.set_ylabel('PnL ($)')

plt.suptitle(f'Simulación P&L — Retorno total: {total_return:+.1f}% | Win rate: {100*win_rate_bt:.1f}%',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'scoring_pnl_simulation.png', bbox_inches='tight')
plt.show()

print("\nNOTA: Esta simulación no incluye comisiones ni impacto de mercado.")
print("El model_score NO es una probabilidad calibrada — interpretar como ranking relativo.")